# Task 1 — Data Loading and Initial Inspection

## Airbnb NYC 2019 listings

This notebook loads the supplied `dataset/AB_NYC_2019.csv` file with pandas and performs a structured first inspection. The source dataset contains Airbnb listings in New York City. The code below keeps the original data unchanged and reports its dimensions, columns, data types, sample records, numeric summaries, and missing values.

> **How to run:** Open this notebook from the coursework project folder and run the cells from top to bottom. The relative CSV path assumes the notebook is in the project root.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
df = pd.read_csv('dataset/AB_NYC_2019.csv')

## 1. Dataset shape

`DataFrame.shape` returns a pair in the order `(rows, columns)`.

In [3]:
rows, columns = df.shape
print(f"Rows: {rows:,}")
print(f"Columns: {columns}")

Rows: 48,895
Columns: 16


## 2. All column names

The following cell lists every field in the CSV in its original order.

In [8]:
print("\nALLColumns:", df.columns.tolist())


ALLColumns: ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']


## 3. Data types

`dtypes` reports the type pandas inferred for each column. Text and date-like fields are currently loaded as strings; `last_review` can be converted to a datetime type later if date analysis is needed.

In [13]:
print("\nData Types:\n", df.dtypes)


Data Types:
 id                                  int64
name                                  str
host_id                             int64
host_name                             str
neighbourhood_group                   str
neighbourhood                         str
latitude                          float64
longitude                         float64
room_type                             str
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                           str
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object


## 4. First five rows

The first five records provide a quick look at how listing information is represented.

In [11]:
print("\nFirst 5 Rows:\n", df.head())


First 5 Rows:
      id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name neighbourhood_group neighbourhood  latitude  longitude  \
0         John            Brooklyn    Kensington  40.64749  -73.97237   
1     Jennifer           Manhattan       Midtown  40.75362  -73.98377   
2    Elisabeth           Manhattan        Harlem  40.80902  -73.94190   
3  LisaRoxanne            Brooklyn  Clinton Hill  40.68514  -73.95976   
4        Laura           Manhattan   East Harlem  40.79851  -73.94399   

         room_type  price  minimum_nights  number_of_reviews last_review  \
0     Private room    149               1       

## 5. Summary statistics for numeric columns

`describe()` gives count, mean, standard deviation, minimum, quartiles, and maximum for each numeric column. The count can be smaller than the number of rows when that numeric field has missing values.

In [7]:
print("\nSummary Statistics:\n", df.describe())


Summary Statistics:
                  id       host_id      latitude     longitude         price  \
count  4.889500e+04  4.889500e+04  48895.000000  48895.000000  48895.000000   
mean   1.901714e+07  6.762001e+07     40.728949    -73.952170    152.720687   
std    1.098311e+07  7.861097e+07      0.054530      0.046157    240.154170   
min    2.539000e+03  2.438000e+03     40.499790    -74.244420      0.000000   
25%    9.471945e+06  7.822033e+06     40.690100    -73.983070     69.000000   
50%    1.967728e+07  3.079382e+07     40.723070    -73.955680    106.000000   
75%    2.915218e+07  1.074344e+08     40.763115    -73.936275    175.000000   
max    3.648724e+07  2.743213e+08     40.913060    -73.712990  10000.000000   

       minimum_nights  number_of_reviews  reviews_per_month  \
count    48895.000000       48895.000000       38843.000000   
mean         7.029962          23.274466           1.373221   
std         20.510550          44.550582           1.680442   
min          1

## 6. Missing values per column

The table reports the number and percentage of missing entries for every column, including columns with no missing data.

In [8]:
print("\nMissing Values Count:\n", df.isnull().sum())


Missing Values Count:
 id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


# Task 2 Data Cleaning and Preparation 

In [14]:
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')
df['reviews_per_month'] = df['reviews_per_month'].fillna(0.0)


print(df[['name', 'host_name', 'reviews_per_month', 'last_review']].isnull().sum())

name                     0
host_name                0
reviews_per_month        0
last_review          10052
dtype: int64


In [15]:
df['last_review'] = pd.to_datetime(df['last_review'])


print(df[['last_review', 'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month']].dtypes)

last_review          datetime64[us]
price                         int64
minimum_nights                int64
number_of_reviews             int64
reviews_per_month           float64
dtype: object


In [16]:
print("Duplicate rows count:", df.duplicated().sum())

Duplicate rows count: 0


In [17]:
invalid_prices_before = (df['price'] <= 0).sum()


df = df[df['price'] > 0]


invalid_prices_after = (df['price'] <= 0).sum()
print(f"Invalid prices before: {invalid_prices_before}, After: {invalid_prices_after}")

Invalid prices before: 11, After: 0


In [18]:
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    
    # Before stats
    print(f"[{column}] Before max: {df[column].max()}, Mean: {df[column].mean():.2f}")
    
    # Cap outliers
    df[column] = df[column].clip(upper=upper_bound)
    
    # After stats
    print(f"[{column}] After max: {df[column].max()}, Mean: {df[column].mean():.2f}")

cap_outliers(df, 'price')
cap_outliers(df, 'minimum_nights')

[price] Before max: 10000, Mean: 152.76
[price] After max: 334, Mean: 133.01
[minimum_nights] Before max: 1250, Mean: 7.03
[minimum_nights] After max: 11, Mean: 3.83


In [19]:
cols_to_drop = ['id', 'host_id', 'name', 'host_name']
df_cleaned = df.drop(columns=cols_to_drop)

print("Columns retained for analysis:", df_cleaned.columns.tolist())
print("Cleaned dataset shape:", df_cleaned.shape)

Columns retained for analysis: ['neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']
Cleaned dataset shape: (48884, 12)
